In [0]:
DECLARE OR REPLACE VARIABLE catalog_use STRING;
DECLARE OR REPLACE VARIABLE schema_use STRING;

In [0]:
SET VAR catalog_use = :catalog_use;
SET VAR schema_use = :schema_use;

In [0]:
USE IDENTIFIER(catalog_use || '.' || schema_use);

In [0]:
SELECT current_catalog(), current_schema();

In [0]:
CREATE TABLE spc_example (
  day_id       INT,
  sample_date  DATE,
  measurement  DOUBLE
);

INSERT INTO spc_example VALUES
  ( 1, DATE '2025-01-01', 10.1),
  ( 2, DATE '2025-01-02',  9.9),
  ( 3, DATE '2025-01-03', 10.3),
  ( 4, DATE '2025-01-04', 10.2),
  ( 5, DATE '2025-01-05',  9.8),
  ( 6, DATE '2025-01-06', 10.0),
  ( 7, DATE '2025-01-07', 10.1),
  ( 8, DATE '2025-01-08',  9.7),
  ( 9, DATE '2025-01-09', 10.4),
  (10, DATE '2025-01-10', 10.2),
  -- in‑control baseline above; now add a few out‑of‑control-ish points
  (11, DATE '2025-01-11', 10.9),
  (12, DATE '2025-01-12',  9.2),
  (13, DATE '2025-01-13', 10.6),
  (14, DATE '2025-01-14',  9.4),
  (15, DATE '2025-01-15', 10.8),
  (16, DATE '2025-01-16',  9.3),
  (17, DATE '2025-01-17', 10.1),
  (18, DATE '2025-01-18',  9.9),
  (19, DATE '2025-01-19', 10.2),
  (20, DATE '2025-01-20', 10.0);


In [0]:
FROM spc_example;

# Individuals/XmR Control Chart

In [0]:
CREATE OR REPLACE FUNCTION spc_x_chart(
	x_col ARRAY<STRUCT<pk BIGINT, x DOUBLE>>
)
RETURNS TABLE (
	pk	BIGINT,
	seq	INT,
	x	DOUBLE,
	cl_x	DOUBLE,
	ucl_x	DOUBLE,
	lcl_x	DOUBLE
)
LANGUAGE SQL
RETURN
	WITH exploded AS (
		SELECT posexplode(x_col) AS (seq, elem)
	),
	stats AS (
		SELECT
			AVG(elem.x)         AS xbar,
			STDDEV_SAMP(elem.x) AS s
		FROM exploded
	),
	limits AS (
		SELECT
			xbar,
			xbar + 3 * s AS ucl_x,
			xbar - 3 * s AS lcl_x
		FROM stats
	)
	SELECT
		e.elem.pk,
		e.seq,
		e.elem.x,
		l.xbar  AS cl_x,
		l.ucl_x,
		l.lcl_x
	FROM exploded e
	CROSS JOIN limits l
	ORDER BY e.seq;

In [0]:
select collect_list(struct(day_id, measurement)) as x_values from spc_example;

In [0]:
select * from spc_x_chart(ARRAY(named_struct('day_id',1,'measurement',10.1), named_struct('day_id',2,'measurement',9.9), named_struct('day_id',3,'measurement',10.3), named_struct('day_id',4,'measurement',10.2), named_struct('day_id',5,'measurement',9.8), named_struct('day_id',6,'measurement',10), named_struct('day_id',7,'measurement',10.1), named_struct('day_id',8,'measurement',9.7), named_struct('day_id',9,'measurement',10.4), named_struct('day_id',10,'measurement',10.2), named_struct('day_id',11,'measurement',10.9), named_struct('day_id',12,'measurement',9.2), named_struct('day_id',13,'measurement',10.6), named_struct('day_id',14,'measurement',9.4), named_struct('day_id',15,'measurement',10.8), named_struct('day_id',16,'measurement',9.3), named_struct('day_id',17,'measurement',10.1), named_struct('day_id',18,'measurement',9.9), named_struct('day_id',19,'measurement',10.2), named_struct('day_id',20,'measurement',10)))

In [0]:
SELECT * FROM spc_example
JOIN LATERAL spc_x_chart(
	(SELECT collect_list(struct(day_id, measurement)) FROM spc_example)
) as chart
ON spc_example.day_id = chart.pk

Databricks visualization. Run in Databricks to view.

# Adaptive Moving Average Control Chart

In [0]:
CREATE OR REPLACE TABLE mavg_example (
	day_id		INT,
	period		INT,
	measurement	DOUBLE
);

WITH base AS (
    SELECT 1 AS period, sequence(1, 50) AS idxs, 10.0 AS mean_val
    UNION ALL
    SELECT 2, sequence(51, 100), 15.0
    UNION ALL
    SELECT 3, sequence(101, 150), 12.0
),
exploded AS (
    SELECT
        period,
        mean_val,
        idx AS day_id
    FROM base
    LATERAL VIEW explode(idxs) t AS idx
),
final AS (
    SELECT
        day_id,
        period,
        ROUND(mean_val + randn() * 1.0, 2) AS measurement
    FROM exploded
)
INSERT INTO mavg_example (day_id, period, measurement)
SELECT day_id, period, measurement
FROM final;

In [0]:
select * from mavg_example;

In [0]:
select collect_list(measurement) as x_values from mavg_example;

In [0]:
-- Replace :window_size with your desired integer before executing the query
CREATE OR REPLACE FUNCTION spc_ewma_chart(
    x_col ARRAY<DOUBLE>,
    lambda_ DOUBLE,
    L DOUBLE
)
RETURNS TABLE (
    seq INT,
    x DOUBLE,
    z_ewma DOUBLE,
    cl_z DOUBLE,
    ucl_z DOUBLE,
    lcl_z DOUBLE
)
LANGUAGE SQL
RETURN
WITH exploded AS (
    SELECT
        pos AS seq,
        col AS x
    FROM posexplode(x_col)
),
ewma_vals_tbl AS (
    SELECT ewma_array(x_col, lambda_) AS ewma_vals
),
ewma_calc AS (
    SELECT
        e.seq,
        e.x,
        ev.ewma_vals[e.seq] AS z_ewma
    FROM exploded e
    CROSS JOIN ewma_vals_tbl ev
),
windowed_stats AS (
    SELECT
        seq,
        x,
        z_ewma,
        AVG(x) OVER (
            ORDER BY seq
            ROWS BETWEEN 9 PRECEDING AND CURRENT ROW
        ) AS mu_hat,
        STDDEV_SAMP(x) OVER (
            ORDER BY seq
            ROWS BETWEEN 9 PRECEDING AND CURRENT ROW
        ) AS sigma_hat
    FROM ewma_calc
)
SELECT
    seq,
    x,
    z_ewma,
    mu_hat AS cl_z,
    mu_hat + L * sigma_hat * SQRT((lambda_ / (2 - lambda_)) * (1 - POWER(1 - lambda_, 2 * (seq + 1)))) AS ucl_z,
    mu_hat - L * sigma_hat * SQRT((lambda_ / (2 - lambda_)) * (1 - POWER(1 - lambda_, 2 * (seq + 1)))) AS lcl_z
FROM windowed_stats
ORDER BY seq

In [0]:
select * from spc_ewma_chart(ARRAY(13.9,15.97,15.15,14.76,14.68,14.81,13.84,15.31,15.9,15.05,14.47,14.35,16.43,16.61,14.95,14.96,15.03,13.03,13.53,16.47,14.73,13.63,15.76,13.73,15.51,13.79,16.48,15.14,14.83,14.45,16.35,14.51,14.17,15.97,15.05,14.8,14.13,14.92,14.39,14.8,15.13,14.04,13.68,15.41,15.48,15.64,15.26,16.19,14.74,15.03,9.59,9.35,8.77,12.01,10.69,9.74,9.18,10.72,8.76,10.74,10.53,11.36,10.52,7.76,10.57,8.53,10.76,9.03,10.79,8.96,10.37,10.21,8.1,9.55,9.64,10.12,10.17,8.63,10.59,9.62,10.73,10.99,11.24,8.9,9.7,10.14,9.79,10.89,10.24,9.28,10.87,10.65,8.79,10.46,8.71,9.08,9.85,11.39,9.83,10.26,10.5,11.21,11.72,12.5,10.76,12.13,13.76,10.61,11.09,13.05,12.69,13.27,10.95,12.46,10.48,11.42,12.25,10.47,12.28,12.12,10.4,10.88,13.88,13.04,10.85,11.21,11.89,11.85,12.14,12.55,11.74,11.78,14.29,12.02,11.36,12.93,13.29,10.09,12.66,11.48,11.37,11,11.14,11.58,13.55,13.82,13.15,12.76,12.09,10.69), 0.2, 3)

Databricks visualization. Run in Databricks to view.